# Streaming output and events

When the expected answer is long and we want to immediately start returning output, we can use live token streams and event callbacks to see progress instantly.

🎞️ The `AgentWorkflow` also supports streaming. The `AgentWorkflow` can be streamed like any other `Workflow`. This works by using the handler that is returned from the workflow. The stream returns a variety of event types as the workflow executes, and we can select which ones to handle.

* If we want to stream the LLM output, we can use the `AgentStream` events, which contain a `delta` of the new output each time
* `AgentInput` events will tell us which agent is running (our current workflow just has one agent)
* `AgentOutput` events will tell us what the agents returned, including which tools they called
* `ToolCall` and `ToolCallResults` will track tools as they are called and their outputs

In this example we're handling the `AgentStream` events. We can tell our handling is working because the output will appear in chunks as we run the cell, rather than appearing all at once.

### ❗️ Note: Run the **hidden cells** below before running the rest of the code. ❗️ 

In [1]:
!pip install llama-index -q -q

In [2]:
!pip install tavily-python -q -q

In [3]:
from tavily import AsyncTavilyClient
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.openai import OpenAI
import os
from openai import OpenAI as OpenAIClient

raw_client = OpenAIClient()

API_KEY   = raw_client.api_key   
API_BASE  = raw_client.base_url

tavily_api_key = os.environ["TAVILY_API_KEY"]

async def search_web(query: str) -> str:
    """Useful for using the web to answer questions."""
    client = AsyncTavilyClient(api_key=tavily_api_key)
    return str(await client.search(query))

llm = OpenAI(model="gpt-4o-mini", api_key=API_KEY, api_base=API_BASE)

workflow = AgentWorkflow.from_tools_or_functions(
    [search_web],
    llm=llm,
    system_prompt="You are a helpful assistant that answers questions. If you don't know the answer, you can search the web for information.",
)

In [6]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

handler = workflow.run(user_msg="What is the weather in Saskatoon?")

async for event in handler.____():
    if isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)

Let's see what's happening in more detail by turning on more logging and running it again.

In [7]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

handler = workflow.run(user_msg="What is the weather in Saskatoon?")

async for event in handler.stream_events():
    if isinstance(event, AgentInput):
       print("Agent input: ", event.input)  # Current input messages
       print("Agent name:", event.current_agent_name)  # Current agent name
    elif isinstance(event, ____):
       print("Agent output: ", event.response)  # Current full response
       print("Tool calls made: ", event.tool_calls)  # Selected tool calls, if any
       print("Raw LLM response: ", event.raw)  # Raw llm api response
    elif isinstance(event, ToolCallResult):
       print("Tool called: ", event.tool_name)  # Tool name
       print("Arguments to the tool: ", event.tool_kwargs)  # Tool kwargs
       print("Tool output: ", event.tool_output)  # Tool output

We can see the agent accepting input, picking the web search tool, calling it with appropriate arguments, and returning the output.